# Recogida sin ruido

Aquest notebook fa la mateixa idea que `dades.ipynb`, pero amb una diferencia important: les imatges son netes, sense pixels propers ni pixels aleatoris.

La mostra de colors es exactament la mateixa que ja tenim a `csv/input_image_sample.csv`. Aixi podem comparar despres si el soroll de la imatge afecta els models.


## Configuracio

Definim les rutes separades. Tot el que generi aquest notebook va a `experiments/sense-soroll/`, per no barrejar-ho amb l'experiment normal.

In [ ]:
from pathlib import Path
import sys
import time

IMAGE_SIZE = 100

# Aquesta part crida l'API i pot gastar diners. Ho deixo apagat per defecte.
RUN_MODEL_QUERIES = True
MODEL_NAMES = ["gpt-4o", "gpt-4o-mini"]
MODEL_TEMPERATURE = 0.2
MODEL_MAX_IMAGES = None  # posa un numero petit, per exemple 5, si vols fer una prova rapida
RETRY_FAILED_MODEL_QUERIES = True

# Canvia a True nomes si vols tornar a crear les imatges netes des de zero.
REGENERATE_IMAGES = False

# Casella de seguretat: esborra nomes fitxers de sense-soroll, no toca l'experiment original.
DELETE_EXISTING_SAMPLE = False


def _format_duration_local(seconds):
    seconds = max(0, float(seconds))
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, remaining_seconds = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {remaining_seconds:04.1f}s"
    hours, remaining_minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(remaining_minutes)}m {remaining_seconds:04.1f}s"


def _install_cell_timer():
    ipython = get_ipython()
    if ipython is None or getattr(ipython, "_color_llm_timer_installed", False):
        return

    def _start_cell_timer(info):
        ipython.user_ns["_cell_started_at"] = time.perf_counter()

    def _stop_cell_timer(result):
        started_at = ipython.user_ns.get("_cell_started_at")
        if started_at is not None:
            elapsed = time.perf_counter() - started_at
            print(f"Temps cel.la: {_format_duration_local(elapsed)}")

    ipython.events.register("pre_run_cell", _start_cell_timer)
    ipython.events.register("post_run_cell", _stop_cell_timer)
    ipython._color_llm_timer_installed = True


_install_cell_timer()

base_dir = Path("recollida-dades")
if not base_dir.exists():
    base_dir = Path("..").resolve()
if not (base_dir / "scripts").exists():
    raise FileNotFoundError(f"No trobo recollida-dades/scripts des de {Path.cwd()}")
source_csv_dir = base_dir / "experiments" / "soroll" / "csv"
source_input_path = source_csv_dir / "input_image_sample.csv"

experiment_dir = base_dir / "experiments" / "sense-soroll"
csv_dir = experiment_dir / "csv"
images_dir = experiment_dir / "images"
tmp_dir = base_dir / "archive" / "tmp" / "sense-soroll-notebook"
logs_dir = experiment_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, tmp_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)
print("Carpeta sense soroll:", experiment_dir)

## Carrega de scripts

Importem les funcions comunes. Aqui necessitem especialment la funcio que genera imatges 100% del color objectiu.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display
from PIL import Image

import utils
importlib.reload(utils)

build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
delete_sample_outputs = utils.delete_sample_outputs
delete_png_files = utils.delete_png_files
generate_solid_images_from_sample = utils.generate_solid_images_from_sample
save_csv = utils.save_csv
save_rgb_sample_grid = utils.save_rgb_sample_grid
save_rgb_sample_map = utils.save_rgb_sample_map
save_chroma_distribution = utils.save_chroma_distribution
save_error_distribution = utils.save_error_distribution
write_log = utils.write_log

required_functions = [
    "generate_solid_images_from_sample",
    "build_final_sample",
    "collect_model_outputs",
    "save_error_distribution",
    "delete_sample_outputs",
    "delete_png_files",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no carregats correctament: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers sense-soroll esborrats: {len(removed)}")

write_log("Notebook sense soroll inicialitzat", logs_dir / "pipeline.log")

## Carrega de la mateixa mostra de colors

No generem colors nous. Llegim la mostra original i en guardem una copia dins `experiments/sense-soroll/csv/` per tenir l'experiment net i separat.

In [ ]:
if not source_input_path.exists():
    raise FileNotFoundError(f"No trobo la mostra original: {source_input_path}")

input_sample = pd.read_csv(source_input_path)
local_input_path = csv_dir / "input_image_sample.csv"
save_csv(input_sample, local_input_path)
write_log(f"Mostra original copiada a sense-soroll: {local_input_path}", logs_dir / "pipeline.log")

print(f"Mostra carregada: {source_input_path}")
print(f"Copia local guardada: {local_input_path}")
print(f"Files: {len(input_sample)}")
display(input_sample.head())

## Mapa de color de control

Aquests grafics son els mateixos que a la mostra normal. Serveixen per comprovar que realment estem treballant amb els mateixos colors.

In [ ]:
color_grid_path = tmp_dir / "rgb_sample_grid.png"
color_map_path = tmp_dir / "rgb_sample_map.png"
chroma_plot_path = tmp_dir / "rgb_chroma_distribution.png"

save_rgb_sample_grid(input_sample, color_grid_path)
save_rgb_sample_map(input_sample, color_map_path)
save_chroma_distribution(input_sample, chroma_plot_path)
write_log(f"Visualitzacions sense-soroll generades a tmp: {color_grid_path}, {color_map_path}, {chroma_plot_path}", logs_dir / "pipeline.log")

print("Graella de les 1000 mostres:")
display(DisplayImage(filename=str(color_grid_path)))

print("Mapa de diagnosi de distribucio:")
display(DisplayImage(filename=str(color_map_path)))

print("Distribucio del chroma:")
display(DisplayImage(filename=str(chroma_plot_path)))

## Generacio de les imatges sense soroll

Aqui generem les imatges netes. Cada PNG te tots els pixels exactament iguals al color de la fila corresponent.

In [ ]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png")}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_IMAGES:
    removed_images = delete_png_files(images_dir)
    if removed_images:
        print(f"Imatges antigues sense-soroll esborrades: {len(removed_images)}")

    image_paths = generate_solid_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        progress=True,
        log_file=logs_dir / "pipeline.log",
    )
    write_log(f"Imatges sense-soroll generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges sense-soroll generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    print(f"Imatges sense-soroll ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]

## Validacio rapida de les imatges

Fem una comprovacio simple: agafem la primera imatge i mirem si tots els pixels son del color esperat.

In [ ]:
first_row = input_sample.iloc[0]
first_image_path = images_dir / first_row["image_name"]
expected_rgb = (int(first_row.r), int(first_row.g), int(first_row.b))

with Image.open(first_image_path) as image:
    pixels = list(image.getdata())

unique_pixels = set(pixels)
print("Imatge revisada:", first_image_path)
print("Color esperat:", expected_rgb)
print("Colors diferents dins la imatge:", len(unique_pixels))

if unique_pixels != {expected_rgb}:
    raise RuntimeError("La imatge no es 100% del color esperat. Alguna cosa no quadra.")

print("Validacio correcta: la imatge es completament uniforme.")

## Consulta als models

Aquesta part usa les imatges sense soroll i desa resultats nomes dins `experiments/sense-soroll/csv/`. Per defecte esta apagada per evitar crides a l'API sense voler.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

utils = importlib.reload(utils)
collect_model_outputs = utils.collect_model_outputs

load_dotenv()

output_path = csv_dir / "outputmodel_image_sample_4o.csv"
model_log_path = logs_dir / "model_calls.log"

if RUN_MODEL_QUERIES:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY. Crea un fitxer .env local amb la clau abans d'executar models.")

    client = OpenAI()
    model_outputs = collect_model_outputs(
        client=client,
        image_paths=image_paths,
        models=MODEL_NAMES,
        temperature=MODEL_TEMPERATURE,
        output_path=output_path,
        log_file=model_log_path,
        max_images=MODEL_MAX_IMAGES,
        retry_failed=RETRY_FAILED_MODEL_QUERIES,
    )

    print(f"Resultats de models sense-soroll guardats: {output_path}")
    print(f"Files totals: {len(model_outputs)}")
    if "status" in model_outputs.columns:
        print("Resum status:")
        print(model_outputs["status"].value_counts(dropna=False))
    display(model_outputs.tail())
else:
    if output_path.exists():
        model_outputs = pd.read_csv(output_path)
        print(f"Models no executats ara. Carrego resultats sense-soroll existents: {len(model_outputs)} files")
        if "status" in model_outputs.columns:
            print("Resum status:")
            print(model_outputs["status"].value_counts(dropna=False))
        display(model_outputs.tail())
    else:
        model_outputs = pd.DataFrame()
        print("Models no executats. Posa RUN_MODEL_QUERIES = True quan vulguis cridar l'API.")

## Dataset final per analisi

Quan hi hagi sortida dels models, unim entrada i resposta igual que al notebook normal, pero escrivint el resultat final a `experiments/sense-soroll/csv/sample-colors_4o.csv`.

In [ ]:
final_path = csv_dir / "sample-colors_4o.csv"
error_plot_path = tmp_dir / "error_cromatic_distribution.png"

if output_path.exists():
    model_outputs = pd.read_csv(output_path)
    final_sample = build_final_sample(input_sample, model_outputs, images_dir=images_dir)
    save_csv(final_sample, final_path)
    write_log(f"Dataset final sense-soroll guardat: {final_path} files={len(final_sample)}", logs_dir / "pipeline.log")

    print(f"Dataset final sense-soroll guardat: {final_path}")
    print(f"Files totals: {len(final_sample)}")

    print("Resum per model i status:")
    display(final_sample.groupby(["model", "status"], dropna=False).size().reset_index(name="files"))

    valid_errors = final_sample[(final_sample["status"] == "ok") & final_sample["error_cromatic"].notna()]
    if valid_errors.empty:
        print("Encara no hi ha prediccions valides per calcular l'error cromatic.")
    else:
        print("Resum error cromatic per model:")
        display(
            valid_errors.groupby("model")["error_cromatic"]
            .agg(["count", "mean", "median", "min", "max"])
            .reset_index()
        )

        save_error_distribution(final_sample, error_plot_path)
        write_log(f"Grafic error cromatic sense-soroll generat: {error_plot_path}", logs_dir / "pipeline.log")
        print("Distribucio de l'error cromatic:")
        display(DisplayImage(filename=str(error_plot_path)))

    display(final_sample.head())
else:
    final_sample = pd.DataFrame()
    print("Encara no existeix outputmodel_image_sample_4o.csv dins sense-soroll. Primer cal executar la cel.la de models.")

## Resultats generats

Comprovem rapidament que tot ha quedat dins la carpeta separada.

In [ ]:
print("CSV sense-soroll:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges sense-soroll:", len(list(images_dir.glob("*.png"))))
print("Fitxers temporals sense-soroll:", sorted(path.name for path in tmp_dir.glob("*.png")))
print("Logs sense-soroll:", sorted(path.name for path in logs_dir.iterdir()))